<a href="https://colab.research.google.com/github/grlee1128/CMPCS-DS-410/blob/main/Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_curve, confusion_matrix
import numpy as np
import warnings
import os

# warnings
warnings.filterwarnings('ignore')

# 1. EFFICIENT DATA LOADING
print("--- Step 1: Loading Data Efficiently ---")

# Define columns we need to avoid Kernel Crash
cols_to_load = [
    'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK',
    'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST',
    'DISTANCE', 'CRS_DEP_TIME', 'DEP_DELAY',
    'CANCELLED', 'DIVERTED'
]

dfs = []
for i in range(1, 16):
    file_path = f"Filtered_part_{i}.csv"
    if os.path.exists(file_path):
        try:
            df_temp = pd.read_csv(file_path, usecols=cols_to_load)
            dfs.append(df_temp)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

if not dfs:
    print("CRITICAL ERROR: No files were loaded.")
else:
    df = pd.concat(dfs, ignore_index=True)
    print(f"Successfully combined files. Shape: {df.shape}")
    # 2. FEATURE ENGINEERING (OPTIMIZED)
    print("\n--- Step 2: Feature Engineering ---")

    # 2a. Filter & Create Target
    # Keep only valid flights
    df = df[(df['CANCELLED'] == 0) & (df['DIVERTED'] == 0)]
    df.dropna(subset=['DEP_DELAY'], inplace=True)

    # Target: 1 if delayed > 15 mins
    y = (df['DEP_DELAY'] > 15).astype(int)

    # 2b. Winter Holiday
    df['winter_holiday'] = 0
    mask_holiday = (df['MONTH'] == 12) | ((df['MONTH'] == 1) & (df['DAY_OF_MONTH'] <= 10))
    df.loc[mask_holiday, 'winter_holiday'] = 1

    # 2c. Time Features
    df['CRS_DEP_HOUR'] = df['CRS_DEP_TIME'] // 100

    bins = [-1, 5, 10, 15, 19, 24]
    labels = ['Late_Night', 'Morning', 'Midday', 'Afternoon_Evening', 'Night']
    df['TIME_OF_DAY'] = pd.cut(df['CRS_DEP_HOUR'], bins=bins, labels=labels, right=True)

    print("Features created.")

    # 3. LABEL ENCODING & PREPARATION
    print("\n--- Step 3: Encoding & Splitting ---")

    # Select Features
    features_to_use = [
        'MONTH', 'DAY_OF_WEEK', 'OP_UNIQUE_CARRIER',
        'ORIGIN', 'DEST', 'DISTANCE',
        'winter_holiday', 'CRS_DEP_HOUR', 'TIME_OF_DAY'
    ]

    X = df[features_to_use].copy()
    cat_cols = ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'TIME_OF_DAY']
    for col in cat_cols:
        X[col] = X[col].astype('category').cat.codes

    # Split Data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Training Data Shape: {X_train.shape}")

    # 4. TRAINING XGBOOST
    print("\n--- Step 4: Training XGBoost ---")

    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        scale_pos_weight=2.5,
        n_estimators=200,
        max_depth=7,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    print("Model trained.")
    # 5. EVALUATION & OPTIMAL THRESHOLD
    print("\n--- Step 5: Finding Optimal Threshold ---")

    y_probs = model.predict_proba(X_test)[:, 1]

    # Calculate scores for all thresholds
    precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

    # Calculate F1 only where valid (avoid divide by zero)
    numerator = 2 * recall * precision
    denominator = recall + precision
    f1_scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator!=0)

    best_idx = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    print(f"Optimal Threshold: {best_thresh:.4f}")
    print(f"Best F1 Score: {best_f1:.4f}")
    # 6. OUTPUT & CONFUSION MATRIX
    print("\n--- Step 6: Results ---")

    # Final Predictions based on optimal threshold
    y_pred_final = (y_probs > best_thresh).astype(int)

    # Classification Report
    print(classification_report(y_test, y_pred_final, target_names=['On-Time', 'Delayed']))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_final)
    print("Confusion Matrix:")
    print(cm)

    tn, fp, fn, tp = cm.ravel()
    print(f"True Negatives (Correct On-Time): {tn}")
    print(f"False Positives (False Alarm): {fp}")
    print(f"False Negatives (Missed Delay): {fn}")
    print(f"True Positives (Caught Delay): {tp}")

    # Save Results to CSV
    results_df = pd.DataFrame({
        'Actual_Delayed': y_test,
        'Predicted_Delayed': y_pred_final,
        'Prediction_Probability': y_probs,
        'Correct_Prediction': (y_test == y_pred_final)
    })

    output_filename = 'confusion_matrix_predictions.csv'
    results_df.to_csv(output_filename, index=False)
    print(f"\nSaved detailed predictions to: {output_filename}")

    # 7. FEATURE IMPORTANCE
    print("\n--- Step 7: Feature Importance ---")
    importance_df = pd.DataFrame({
        'feature': X.columns,
        'importance': model.feature_importances_
    }).sort_values(by='importance', ascending=False)

    print(importance_df)
    print("\n--- Pipeline Complete ---")

RANDOM FOREST


PARAMETER TUNING

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
import numpy
import time
import random
import csv

# DATA PROCESSING
df1 = ss.read.csv("/storage/home/pvm5528/work/project/part-00000-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df2 = ss.read.csv("/storage/home/pvm5528/work/project/part-00001-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df3 = ss.read.csv("/storage/home/pvm5528/work/project/part-00002-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df4 = ss.read.csv("/storage/home/pvm5528/work/project/part-00003-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df5 = ss.read.csv("/storage/home/pvm5528/work/project/part-00004-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df5 = ss.read.csv("/storage/home/pvm5528/work/project/part-00005-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df = df1.union(df2).union(df3).union(df4).union(df5)

df = df.drop("CRS_DEP_TIME","DEP_TIME", "ARR_TIME", "CRS_ARR_TIME", "DIVERTED","ARR_DELAY", "FL_DATE", "ORIGIN_CITY_NAME", "DEST_CITY_NAME", "CANCELLATION_CODE", "CANCELLED", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY")
df = df.withColumn("christmas_season", ((col("MONTH") == 12) | ((col("MONTH") == 1) & (col("DAY_OF_MONTH").between(1, 10)))).cast("int"))
df = df.fillna(0)
df = df.repartition(32)

# MODEL LABELS
df = df.withColumn("label", col("DEP_DELAY").cast("double"))
df = df.withColumn("classifier_label", when(col("DEP_DELAY") > 15, 1).otherwise(0))
df = df.drop("DEP_DELAY")

# FEATURE SET UP
strCols = ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST"]
vectorCols = [col for col in df.columns if col not in strCols + ["label", "classifier_label"]]

indexers = [StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep") for col in strCols]
assembler = VectorAssembler(inputCols=[f"{col}_idx" for col in strCols] + vectorCols, outputCol="features")

# METRIC EVALUATORS
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
f1_classifier = MulticlassClassificationEvaluator(labelCol="classifier_label", predictionCol = "prediction", metricName="f1")
precision_classifier = MulticlassClassificationEvaluator(labelCol="classifier_label", predictionCol = "prediction", metricName="weightedPrecision")
recall_classifier = MulticlassClassificationEvaluator(labelCol="classifier_label", predictionCol = "prediction", metricName="weightedRecall")
accuracy_classifier = MulticlassClassificationEvaluator(labelCol="classifier_label", predictionCol = "prediction", metricName="accuracy")

# DATA SPLIT / PIPELINE SETUP
df_train, df_val, df_test = df.randomSplit([.6,.2,.2], seed=42)
pipeline= Pipeline(stages=indexers + [assembler])
model = pipeline.fit(df_train)
train_set = model.transform(df_train)
val_set = model.transform(df_val)
test_set = model.transform(df_test)

# HYPERPARAM TUNING
numTrees_list = list(range(30,151,30))
maxDepth_list = list(range(5,13))

best_reg_model = None
best_classifier_model = None

best_rmse = float("inf")
best_f1 = float("-inf")

best_reg_config = None
best_classifier_config = None

print("message: HPT FOR DELAY")
for numTrees in numTrees_list:
    print(f"message: trees: {numTrees}")
    for maxDepth in maxDepth_list:
        rf = RandomForestRegressor(
            featuresCol="features",
            labelCol="label",
            numTrees=numTrees,
            maxDepth=maxDepth,
            maxBins=256,
            seed=42
        )

        rf_classifier = RandomForestClassifier(
            featuresCol="features",
            labelCol="classifier_label",
            numTrees=numTrees,
            maxDepth=maxDepth,
            maxBins=256,
            seed=42
        )

        rf_model = rf.fit(train_set)
        rf_val_pred = rf_model.transform(val_set)
        rmse = evaluator.evaluate(rf_val_pred)

        if rmse < best_rmse:
            best_rmse = rmse
            best_reg_model = rf_model
            best_reg_config = (numTrees, maxDepth)

        rf_classifier_model = rf_classifier.fit(train_set)
        rf_classifier_val_pred = rf_classifier_model.transform(val_set)
        f1 = f1_classifier.evaluate(rf_classifier_val_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_classifier_model = rf_classifier_model
            best_classifier_config = (numTrees, maxDepth)

# RESUlTS
results = [best_reg_config[0], best_reg_config[1], best_rmse, best_classifier_config[0], best_classifier_config[1], best_f1]

with open("hyperparam.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Regression numTrees", "Regression maxDepth", "Best RMSE", "Classifier numTrees", "Classifier maxDepth", "Best f1"])
    writer.writerow(results)

# TEST SET FOR REGRESSION
test_pred_reg = best_reg_model.transform(test_set)
test_rmse = evaluator.evaluate(test_pred_reg)
r2 = evaluator.evaluate(test_pred_reg, {evaluator.metricName: "r2"})

# TEST SET FOR CLASSIFIER
test_pred_classifier = best_classifier_model.transform(test_set)
test_f1 = f1_classifier.evaluate(test_pred_classifier)
test_precision = precision_classifier.evaluate(test_pred_classifier)
test_recall = recall_classifier.evaluate(test_pred_classifier)
test_acc = accuracy_classifier.evaluate(test_pred_classifier)

# TEST SET RESULTS
results = [test_rmse, r2, test_f1, test_precision, test_recall, test_acc]

with open("test_results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Test RMSE", "R^2", "Test f1", "Test precision", "Test recall", "Test accurary"])
    writer.writerow(results)

# TREE PARSER
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"message: Execution time: {execution_time:.4f} seconds")

TEST RESULTS

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
import time
import random
import csv

start_time = time.perf_counter()

# DATA PROCESSING
df1 = ss.read.csv("/storage/home/pvm5528/work/project/part-00000-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df2 = ss.read.csv("/storage/home/pvm5528/work/project/part-00001-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df3 = ss.read.csv("/storage/home/pvm5528/work/project/part-00002-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df4 = ss.read.csv("/storage/home/pvm5528/work/project/part-00003-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df5 = ss.read.csv("/storage/home/pvm5528/work/project/part-00004-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df5 = ss.read.csv("/storage/home/pvm5528/work/project/part-00005-dfe6b37f-71f7-4a4a-8ac3-39d77bdc77a7-c000.csv", header=True, inferSchema=True)
df = df1.union(df2).union(df3).union(df4).union(df5)

df = df.drop("CRS_DEP_TIME","DEP_TIME", "ARR_TIME", "CRS_ARR_TIME", "DIVERTED","ARR_DELAY", "FL_DATE", "ORIGIN_CITY_NAME", "DEST_CITY_NAME", "CANCELLATION_CODE", "CANCELLED", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY")
df = df.withColumn("christmas_season", ( (col("MONTH") == 12) | ( (col("MONTH") == 1) & (col("DAY_OF_MONTH").between(1, 10)) ) ).cast("int"))
df = df.fillna(0)
df = df.repartition(32)

# MODEL LABELS
df = df.withColumn("label", col("DEP_DELAY").cast("double"))
df = df.withColumn("classifier_label", when(col("DEP_DELAY") > 15, 1).otherwise(0))
df = df.drop("DEP_DELAY")

# FEATURE SET UP
strCols = ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST"]
vectorCols = [col for col in df.columns if col not in strCols + ["label", "classifier_label"]]

indexers = [StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep") for col in strCols]
assembler = VectorAssembler(inputCols=[f"{col}_idx" for col in strCols] + vectorCols, outputCol="features")

# METRIC EVALUATORS
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
f1_classifier = MulticlassClassificationEvaluator(labelCol="classifier_label", predictionCol = "prediction", metricName="f1")
precision_classifier = MulticlassClassificationEvaluator(labelCol="classifier_label", predictionCol = "prediction", metricName="weightedPrecision")
recall_classifier = MulticlassClassificationEvaluator(labelCol="classifier_label", predictionCol = "prediction", metricName="weightedRecall")
accuracy_classifier = MulticlassClassificationEvaluator(labelCol="classifier_label", predictionCol = "prediction", metricName="accuracy")

# SPLIT DF / PIPELINE SETUP
df_train_full, df_test = df.randomSplit([0.8, 0.2], seed=42)
preproc_pipeline = Pipeline(stages = indexers + [assembler])
preproc_model = preproc_pipeline.fit(df_train_full)
preproc_train_full = preproc_model.transform(df_train_full)
preproc_test = preproc_model.transform(df_test)

# REGRESSION MODEL (USING HYPERPARAMS*83.3% FROM hyperparam.csv)
rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="label",
        numTrees=75,
        maxDepth=10,
        maxBins=256,
)

# CLASSIFICATION MODEL
rf_classifier = RandomForestClassifier(
        featuresCol="features",
        labelCol="classifier_label",
        numTrees=50,
        maxDepth=10,
        maxBins=256,
)

NUM_TRIALS = 50
results = []
classifier_results = []
all_importances = None

for trial in range(NUM_TRIALS):

    # SEED SETUP
    seed = random.randint(1,999)
    df_train, df_val = preproc_train_full.randomSplit([.75, .25], seed=seed)

    # REGRESSION MODEL
    rf_model = rf.fit(df_train)
    val_pred = rf_model.transform(df_val)

    val_rmse = evaluator.evaluate(val_pred)
    val_r2 = evaluator.evaluate(val_pred, {evaluator.metricName: "r2"})

    # CLASSIFICATION MODEL
    classifier_model = rf_classifier.fit(df_train)
    classifier_val_pred = classifier_model.transform(df_val)

    f1 = f1_classifier.evaluate(classifier_val_pred)
    precision = precision_classifier.evaluate(classifier_val_pred)
    recall = recall_classifier.evaluate(classifier_val_pred)
    acc = accuracy_classifier.evaluate(classifier_val_pred)

    # TRAIL RESULTS
    results.append((trial+1, seed, val_rmse, val_r2))
    classifier_results.append((trial+1,seed,f1,precision,recall,acc))

    # FEATURE IMPORTANCE
    importances = rf_model.featureImportances.toArray()
    if all_importances is None:
        all_importances = importances
    else:
        all_importances += importances
    print(f"message: {trial+1} done")

# BEST SEED SELECTION
best_score = float("-inf")
best_seed = None
best_stats = None
for ((trial, seed, rmse, r2), (_, _, f1, prec, rec, acc)) in zip(results, classifier_results):

    score = (1/rmse + 1e-6) + f1 + prec + rec + acc
    if score > best_score:
        best_score = score
        best_seed = seed
        best_stats = (rmse, f1, prec, rec, acc)
print("score done")

# TEST ON BEST MODEL
# RESPLIT DATA FOR BEST SEED
final_rf = rf.fit(preproc_train_full)
classifier_final_rf = rf_classifier.fit(preproc_train_full)

test_pred = final_rf.transform(preproc_test)
classifier_test_pred = classifier_final_rf.transform(preproc_test)

# EXTRACT BEST MODEL TEST METRICS
test_rmse = evaluator.evaluate(test_pred)
test_r2 = evaluator.evaluate(test_pred, {evaluator.metricName: "r2"})
test_f1 = f1_classifier.evaluate(classifier_test_pred)
test_precision = precision_classifier.evaluate(classifier_test_pred)
test_recall = recall_classifier.evaluate(classifier_test_pred)
test_acc = accuracy_classifier.evaluate(classifier_test_pred)

# CSV RESULTS
with open("reg_results_50t_final.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["trial", "seed", "rmse", "r2"])
    writer.writerows(results)

with open("classifier_results_50t_final.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["trial", "seed", "f1", "precision", "recall", "accuracy"])
    writer.writerows(classifier_results)

model_assembler = preproc_model.stages[-1]
features= model_assembler.getInputCols()
mean_importance = all_importances / NUM_TRIALS
feat_imp_zip = list(zip(features, mean_importance))
sorted_importances = sorted(feat_imp_zip, key=lambda x: x[1], reverse=True)

with open("feature_importance_50t_final.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["feature", "importance"])
    for feature, important in sorted_importances:
        writer.writerow([feature, float(important)])

with open("test_results_50t_final.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Test RMSE", "R^2", "Test f1", "Test precision", "Test recall", "Test accurary"])
    writer.writerow([test_rmse, test_r2, test_f1, test_precision, test_recall, test_acc])

# PREDICTIONS FOR CLASSIFIER
predictions = classifier_test_pred.withColumn("prediction_int", col("prediction").cast("int"))
predictions = predictions.withColumn("TP", when((col("classifier_label") == 1) & (col("prediction_int") == 1), 1).otherwise(0))
predictions = predictions.withColumn("FP", when((col("classifier_label") == 0) & (col("prediction_int") == 1), 1).otherwise(0))
predictions = predictions.withColumn("TN", when((col("classifier_label") == 0) & (col("prediction_int") == 0), 1).otherwise(0))
predictions = predictions.withColumn("FN", when((col("classifier_label") == 1) & (col("prediction_int") == 0), 1).otherwise(0))

counts = predictions.agg(
    spark_sum("TP"),
    spark_sum("FP"),
    spark_sum("TN"),
    spark_sum("FN")
).collect()[0]

TP = counts["sum(TP)"]
FP = counts["sum(FP)"]
TN = counts["sum(TN)"]
FN = counts["sum(FN)"]
results = [TP, FP, TN, FN]

with open("PREDICTIONS_50t_final.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["TP", "FP", "TN", "FN"])
    writer.writerow(results)